# Prepare splits for feature selection

Read the engineered cases and eligible feature list from notebooks 3 and 4, respectively.

Keep the case key, time fields and target separate from the model inputs.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

processed_dir = Path("dataset/processed")
if not processed_dir.is_dir():
    processed_dir = Path("../dataset/processed")

engineered_path = processed_dir / "train_base_engineered.parquet"
inventory_path = processed_dir / "model_feature_eligibility.csv"
if not engineered_path.is_file() or not inventory_path.is_file():
    raise FileNotFoundError("Run notebooks 3 and 4 to generate the engineered table and inventory.")

# Load the inventory and identify eligible features, metadata columns, and the label column
inventory = pd.read_csv(inventory_path)
eligible_features = inventory.loc[inventory["role"].eq("eligible"), "column_name"].tolist()
metadata_columns = ["case_id", "date_decision", "WEEK_NUM", "MONTH"]
label_column = "target"
input_columns = [*metadata_columns, label_column, *eligible_features]
if not eligible_features or len(input_columns) != len(set(input_columns)):
    raise ValueError("The inventory must contain distinct eligible features separate from metadata and target.")

# Load the engineered table and filter to the input columns, then identify categorical columns
con = duckdb.connect(config={"threads": 2})
engineered = con.read_parquet(str(engineered_path))
model_data = engineered.select(*(duckdb.ColumnExpression(name) for name in input_columns))

column_types = dict(zip(model_data.columns, map(str, model_data.types)))
categorical_columns = [name for name in eligible_features if column_types[name] == "VARCHAR"]

## Split by week

Use approximately `70%` of the observed weeks for training, the next `15%` for validation,
and the latest `15%` for the local test set.
- The proportions are based on week counts. Each week stays together.

In [4]:
weeks = [
    week for (week,) in model_data.project("WEEK_NUM").distinct().order("WEEK_NUM").fetchall()
]
train_end = int(len(weeks) * 0.70)
val_end = int(len(weeks) * 0.85)
if None in weeks or not 0 < train_end < val_end < len(weeks):
    raise ValueError("Expected enough non-null weeks for three chronological splits.")

train_weeks = weeks[:train_end]
val_weeks = weeks[train_end:val_end]
test_weeks = weeks[val_end:]


def load_split(split_weeks):
    first_week, last_week = split_weeks[0], split_weeks[-1]
    frame = model_data.filter(f"WEEK_NUM BETWEEN {first_week} AND {last_week}").df()
    return frame[eligible_features], frame[label_column], frame[metadata_columns]


X_train, y_train, train_meta = load_split(train_weeks)
X_val, y_val, val_meta = load_split(val_weeks)
X_test, y_test, test_meta = load_split(test_weeks)

split_summary = pd.DataFrame(
    [
        ("train", train_weeks[0], train_weeks[-1], len(train_weeks), len(X_train)),
        ("validation", val_weeks[0], val_weeks[-1], len(val_weeks), len(X_val)),
        ("test", test_weeks[0], test_weeks[-1], len(test_weeks), len(X_test)),
    ],
    columns=["split", "first_week", "last_week", "weeks", "cases"],
)
display(split_summary)

,split,first_week,last_week,weeks,cases
0,train,0,63,64,1282082
1,validation,64,77,14,92870
2,test,78,91,14,151707


## Prepare LightGBM categories

LightGBM natively handles categorical features. A small bit of processing is still required though, for the categorical features. For a given categorical feature,
1. Use the training to convert the data type of the column from `str`/`object` to `category`. Pandas will then learn the existing categories in the training set.
2. Use the learned categories to convert the validation and testing dataset. A label absent from training becomes missing in those splits.

In [5]:
for column in categorical_columns:
    X_train[column] = X_train[column].astype("category")
    training_dtype = X_train[column].dtype
    X_val[column] = X_val[column].astype(training_dtype)
    X_test[column] = X_test[column].astype(training_dtype)

category_summary = pd.DataFrame({
    "column_name": categorical_columns,
    "training_categories": [len(X_train[name].cat.categories) for name in categorical_columns],
})
display(category_summary)

,column_name,training_categories
0,person_1_applicant_empl_industry_691L,24
1,person_1_applicant_incometype_1044T,9


## Compare feature groups

Start with a LightGBM baseline using all eligible features. Then retrain with one
related group removed at a time. Examples include the recorded loan terms and their
ratios, the debit card opening dates, and the prior-application date summaries.
Compare weekly Gini and the stability score on the same validation weeks with the
same model settings. Keep the full set if removing groups gives no meaningful benefit.